In [1]:
import sys
dir_git = "/Users/usuario/git/sisepuede"
if dir_git not in sys.path:
    sys.path.append(dir_git)


import importlib
import matplotlib.pyplot as plt
import numpy as np
import os, os.path
import pandas as pd
import pathlib
import sisepuede.core.support_classes as sc
import sisepuede.manager.sisepuede_file_structure as sfs
import sisepuede.manager.sisepuede_models as sm
import sisepuede.utilities.data_support._elasticities as elast
import sisepuede.utilities._toolbox as sf
import utils.common_data_needs as cdn
import warnings
warnings.filterwarnings("ignore")

from typing import *

plt.style.use("dark_background", )



/Users/usuario/git/sisepuede/sisepuede/utilities/_toolbox.py:533: UserWarning: Path '/Users/usuario/git/sisepuede/sisepuede/out/sisepuede_run_2025-10-26T20;06;12.879929' not found. It will not be created.
  warnings.warn(msg)
/Users/usuario/git/sisepuede/sisepuede/core/model_attributes.py:6830: UserWarning: 

                        MISSIONSEARCHNOTE: As of 2023-10-06, there is a temporary solution 
                        implemeted in ModelAttributes.get_variable_to_simplex_group_dictionary() 
                        to ensure that transition probability rows are enforced on a simplex.
                        
                        
                        FIX THIS ASAP TO DERIVE PROPERLY.
                        
                        
  warnings.warn(
/Users/usuario/git/sisepuede/sisepuede/utilities/_toolbox.py:533: UserWarning: Path '/Users/usuario/git/sisepuede/sisepuede/out/sisepuede_run_2025-10-26T20;06;13.208772' not found. It will not be created.
  warnings.warn(msg)
/Use

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


Precompiling NemoMod...
Info Given NemoMod was explicitly requested, output will be shown live 
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
   1358.1 ms  ? NemoMod
[ Info: Precompiling NemoMod [a3c327a0-d2f0-11e8-37fd-d12fd35c3c72] 
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Info: Skipping precompilation due to precompilable error. Importing NemoMod [a3c327a0-d2f0-11e8-37fd-d12fd35c3c72].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.
/Users/usuario/git/sisepuede/sisepuede/utilities/_toolbox.py:2633: UserWarning: Warning passed from optional_log: Successfully initialized JuMP optimizer from solver module HiGHS..
  warnings.warn(f"Warning passed from optional_log: {msg}.")


# Initialize some SISEPUEDE components

In [39]:
dict_ssp = cdn._setup_sisepuede_elements()

matt = dict_ssp.get("model_attributes", )
models = dict_ssp.get("models", )
regions = dict_ssp.get("regions", )
time_periods = dict_ssp.get("time_periods", )

# setup region
_REGION_NAME = "uganda"
_REGION_ISO = regions.return_region_or_iso(_REGION_NAME, return_type = "iso")

# set year of survey data
_YEAR_START = 2015
_YEAR_SURVEY = 2021
_YEAR_TARGET = 2100
_DF_YEARS = cdn.spawn_years_space_df((_YEAR_START, _YEAR_TARGET + 1))


##################################################
#    initialize a global dict for populations    #
##################################################

_ATTRIBUTE_TABLE_LVST = matt.get_attribute_table(matt.subsec_name_lvst, )

# some model variables
_MODVAR_LVST_ANIMAL_MASS = matt.get_variable(models.model_afolu.modvar_lvst_animal_weight, )
_MODVAR_LVST_GENFACTOR_N = matt.get_variable(models.model_afolu.modvar_lvst_genfactor_nitrogen, )
_MODVAR_LVST_GENFACTOR_VS = matt.get_variable(models.model_afolu.modvar_lvst_genfactor_volatile_solids, )

# output file names
_FN_ANIMAL_MASS = cdn.file_name_from_variable(_MODVAR_LVST_ANIMAL_MASS, )
_FN_GENFACTOR_N = cdn.file_name_from_variable(_MODVAR_LVST_GENFACTOR_N, )
_FN_GENFACTOR_VS = cdn.file_name_from_variable(_MODVAR_LVST_GENFACTOR_VS, )



# load without data that will be built here
df_uganda = cdn._build_from_outputs(
    (
        min(time_periods.all_years),
        max(time_periods.all_years)
    ),
    fns_exclude = [
        _FN_ANIMAL_WEIGHT,
        _FN_GENFACTOR_N,
        _FN_GENFACTOR_VS
    ],
    force_complete_build = True,
    merge_type = "outer",
    print_info = False,
    stop_on_error = True, 
)

df_uganda = time_periods.tps_to_years(df_uganda, )


# build a generic function
def df_out_from_dict(
    dict_new: Dict[str, Any],
    df_base: pd.DataFrame,
    modvar: Union[str, 'ModelVariable'],
) -> pd.DataFrame:
    """DataFrame from dictionary values
    """
    
    df_out = (
        modvar
        .get_from_dataframe(
            df_base,
            fields_additional = [time_periods.field_year],
        )
        .copy()
    )

    for k, v in dict_new.items():
        field = modvar.build_fields(
            category_restrictions = k,
        )
        
        if field in df_out.columns:
            df_out[field] = v

    return df_out
    


## Use animal mass and generation factor from BUR


![Change in improved over time](./input_data/bur/table_2.46.png)



###  Show the base values for animal mass

In [23]:
_MODVAR_LVST_ANIMAL_MASS.get_from_dataframe(
    df_uganda,
).head()

,avgmass_lvst_animal_buffalo_kg,avgmass_lvst_animal_cattle_dairy_kg,avgmass_lvst_animal_cattle_nondairy_kg,avgmass_lvst_animal_chickens_kg,avgmass_lvst_animal_goats_kg,avgmass_lvst_animal_horses_kg,avgmass_lvst_animal_mules_kg,avgmass_lvst_animal_pigs_kg,avgmass_lvst_animal_sheep_kg
0,315.0,508.0,303.0,1.1,24.0,238.0,130.0,65.0,31.0
1,315.0,508.0,303.0,1.1,24.0,238.0,130.0,65.0,31.0
2,315.0,508.0,303.0,1.1,24.0,238.0,130.0,65.0,31.0
3,315.0,508.0,303.0,1.1,24.0,238.0,130.0,65.0,31.0
4,315.0,508.0,303.0,1.1,24.0,238.0,130.0,65.0,31.0


In [41]:
# mass conversion
um_mass = matt.get_unit("mass")
conv_factor = um_mass.convert(
    "kg",
    _MODVAR_LVST_ANIMAL_MASS.attribute("mass")
)

# build dictionary of diffrent vals by cat
dict_animal_mass = {
    #"buffalo": ,
    "cattle_dairy": 275,
    "cattle_nondairy": 173,
    "chickens": 1.8,
    "goats": 30,
    #"horses": ,
    #"mules": ,
    "pigs": 28,
    "sheep": 28,
}

dict_animal_mass = dict((k, v*conv_factor) for k, v in dict_animal_mass.items())


df_animal_mass = df_out_from_dict(
    dict_animal_mass,
    df_uganda,
    _MODVAR_LVST_ANIMAL_MASS,
)

## Build N Generation Factor


In [53]:

# build dictionary of diffrent vals by cat
dict_n_generation = {
    #"buffalo": ,
    "cattle_dairy": 0.0006,
    "cattle_nondairy": 0.00063,
    "chickens": 0.00082,
    "goats": 0.00137,
    #"horses": ,
    #"mules": ,
    "pigs": 0.00106,
    "sheep": 0.0017,
}

dict_animal_mass = dict((k, v*conv_factor) for k, v in dict_animal_mass.items())

# get some scalars, which will be used to adjust volatile solids
dict_scale_vs = {}
for k, v in dict_n_generation.items():
    field = _MODVAR_LVST_GENFACTOR_N.build_fields(
        category_restrictions = k
    )
    scalar = v/(df_uganda[field].iloc[0])

    dict_scale_vs.update({k: scalar, })


# n generation numbers
df_n_generation = df_out_from_dict(
    dict_n_generation,
    df_uganda,
    _MODVAR_LVST_GENFACTOR_N,
)

In [62]:
sf._write_csv(
    df_animal_mass,
    cdn._PATH_OUTPUTS.joinpath(_FN_ANIMAL_MASS)
)

sf._write_csv(
    df_n_generation,
    cdn._PATH_OUTPUTS.joinpath(_FN_GENFACTOR_N)
)


True